# Winger Shortlists V2

Role scores using progressive metrics, dribble quality, age cohorts, and form trend.

In [32]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../data/processed")
FEATURES_PATH = DATA_DIR / "winger_features_v2.parquet"
STABILITY_PATH = DATA_DIR / "winger_player_cluster_stability.parquet"  # optional

TOP_N = 50
MIN_MINUTES = 300


In [33]:
df = pd.read_parquet(FEATURES_PATH)
df = df[df["minutes"] >= MIN_MINUTES].copy()

# Merge cluster labels from profiles so dashboard filters work
profiles_path = DATA_DIR / "winger_profiles.parquet"
if profiles_path.exists():
    prof = pd.read_parquet(profiles_path)
    keep = [c for c in ["player_id", "cluster", "cluster_name"] if c in prof.columns]
    if "player_id" in keep:
        prof = prof[keep].drop_duplicates(subset=["player_id"])
        df = df.merge(prof, on="player_id", how="left")

if "cluster" not in df.columns:
    df["cluster"] = -1
if "cluster_name" not in df.columns:
    df["cluster_name"] = "Unknown"

df["cluster"] = pd.to_numeric(df["cluster"], errors="coerce").fillna(-1).astype(int)
df["cluster_name"] = df["cluster_name"].fillna("Unknown").astype(str)

# Merge stability
if STABILITY_PATH.exists():
    st = pd.read_parquet(STABILITY_PATH)[["player_id", "cluster_agreement"]].drop_duplicates(subset=["player_id"])
    df = df.merge(st, on="player_id", how="left")
else:
    df["cluster_agreement"] = 1.0

df["cluster_agreement"] = df["cluster_agreement"].fillna(1.0)
print("Eligible players:", len(df))
print("Cluster names:", sorted(df["cluster_name"].dropna().unique().tolist())[:10])
df.head()


Eligible players: 1438
Cluster names: ['1v1 Specialist', 'Inside Forward Finisher']


,player_id,dribbles_attempted,dribbles_successful,carries,passes,shots,xg,minutes,progressive_passes,progressive_carries,...,dribbles_per90_trend_last5_minus_season,carries_per90_trend_last5_minus_season,passes_per90_trend_last5_minus_season,shots_per90_trend_last5_minus_season,xg_per90_trend_last5_minus_season,age_cohort,player_name,cluster,cluster_name,cluster_agreement
0,2941,30.0,15.0,236.0,177.0,21.0,3.395412,1040.0,23.0,0.0,...,-9.346154e-01,-1.869231,-1.471154,-0.294231,0.007976,Unknown,Ismaïla Sarr,0,Inside Forward Finisher,0.533333
1,2944,8.0,5.0,87.0,130.0,4.0,0.351674,1170.0,15.0,0.0,...,7.692308e-02,2.169231,1.769231,-0.030769,-0.000828,Unknown,Benjamin Corgnet,1,1v1 Specialist,0.533333
2,2948,12.0,8.0,101.0,76.0,7.0,1.190467,780.0,11.0,0.0,...,-2.220446e-16,1.084615,0.646154,0.023077,-0.009435,Unknown,Nabil Fekir,1,1v1 Specialist,0.533333
3,2950,15.0,8.0,350.0,388.0,10.0,1.642851,910.0,67.0,0.0,...,3.164835e-01,0.692308,-0.712088,-0.296703,-0.123699,Unknown,Jonas Martin,0,Inside Forward Finisher,0.533333
4,2955,72.0,44.0,1580.0,2137.0,142.0,12.233072,4030.0,569.0,0.0,...,-8.486352e-02,-1.223821,-5.355335,-1.232754,-0.090596,Unknown,Ronny Rodelin,0,Inside Forward Finisher,0.533333


In [34]:
# Build percentile features
metric_cols = [
    "dribbles_attempted_per90",
    "dribbles_successful_per90",
    "dribble_success_rate",
    "carries_per90",
    "progressive_carries_per90",
    "passes_per90",
    "progressive_passes_per90",
    "shots_per90",
    "xg_per90",
    "dribbles_per90_trend_last5_minus_season",
    "shots_per90_trend_last5_minus_season",
    "xg_per90_trend_last5_minus_season",
]

for c in metric_cols:
    if c not in df.columns:
        df[c] = 0.0
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    df[f"{c}_pct"] = df[c].rank(pct=True) * 100

df.head()


,player_id,dribbles_attempted,dribbles_successful,carries,passes,shots,xg,minutes,progressive_passes,progressive_carries,...,dribble_success_rate_pct,carries_per90_pct,progressive_carries_per90_pct,passes_per90_pct,progressive_passes_per90_pct,shots_per90_pct,xg_per90_pct,dribbles_per90_trend_last5_minus_season_pct,shots_per90_trend_last5_minus_season_pct,xg_per90_trend_last5_minus_season_pct
0,2941,30.0,15.0,236.0,177.0,21.0,3.395412,1040.0,23.0,0.0,...,45.653686,60.465925,50.034771,41.446453,27.607789,85.361613,94.506259,7.371349,15.994437,79.137691
1,2944,8.0,5.0,87.0,130.0,4.0,0.351674,1170.0,15.0,0.0,...,73.191933,13.873435,50.034771,23.157163,13.838665,16.342142,23.296245,79.172462,28.511822,29.415855
2,2948,12.0,8.0,101.0,76.0,7.0,1.190467,780.0,11.0,0.0,...,79.972184,28.511822,50.034771,18.984701,16.342142,45.201669,73.435327,31.397775,75.034771,24.617524
3,2950,15.0,8.0,350.0,388.0,10.0,1.642851,910.0,67.0,0.0,...,55.180807,91.029207,50.034771,93.184979,82.058414,56.675939,80.528512,88.038943,15.855355,3.059805
4,2955,72.0,44.0,1580.0,2137.0,142.0,12.233072,4030.0,569.0,0.0,...,70.271210,91.724618,50.034771,98.609179,97.844228,97.913769,93.045897,27.399166,1.460362,5.841446


In [35]:
# Adjustment factors
age_factor_map = {
    "U21": 1.05,
    "22-24": 1.08,
    "25-27": 1.00,
    "28-30": 0.94,
    "31+": 0.88,
    "Unknown": 1.00,
}

if "age_cohort" not in df.columns:
    df["age_cohort"] = "Unknown"

df["age_factor"] = df["age_cohort"].astype(str).map(age_factor_map).fillna(1.0)

trend_signal = (
    0.4 * df["dribbles_per90_trend_last5_minus_season_pct"] +
    0.3 * df["shots_per90_trend_last5_minus_season_pct"] +
    0.3 * df["xg_per90_trend_last5_minus_season_pct"]
)

df["form_factor"] = (0.9 + 0.2 * (trend_signal / 100.0)).clip(lower=0.90, upper=1.10)
df[["age_cohort", "age_factor", "form_factor"]].head()


,age_cohort,age_factor,form_factor
0,Unknown,1.0,0.962976
1,Unknown,1.0,0.998095
2,Unknown,1.0,0.984910
3,Unknown,1.0,0.981780
4,Unknown,1.0,0.926300


In [36]:
# Role scoring
df["score_direct_runner"] = (
    0.30 * df["dribbles_attempted_per90_pct"] +
    0.15 * df["dribbles_successful_per90_pct"] +
    0.10 * df["dribble_success_rate_pct"] +
    0.20 * df["carries_per90_pct"] +
    0.20 * df["progressive_carries_per90_pct"] +
    0.05 * df["progressive_passes_per90_pct"]
)

df["score_playmaker_winger"] = (
    0.25 * df["passes_per90_pct"] +
    0.35 * df["progressive_passes_per90_pct"] +
    0.10 * df["dribble_success_rate_pct"] +
    0.15 * df["carries_per90_pct"] +
    0.10 * df["progressive_carries_per90_pct"] +
    0.05 * df["xg_per90_pct"]
)

df["score_inside_forward"] = (
    0.30 * df["shots_per90_pct"] +
    0.30 * df["xg_per90_pct"] +
    0.15 * df["dribbles_attempted_per90_pct"] +
    0.10 * df["dribble_success_rate_pct"] +
    0.10 * df["progressive_carries_per90_pct"] +
    0.05 * df["carries_per90_pct"]
)

for col in ["score_direct_runner", "score_playmaker_winger", "score_inside_forward"]:
    df[f"{col}_adj"] = df[col] * df["age_factor"] * df["form_factor"] * df["cluster_agreement"]

df[["player_id", "score_direct_runner_adj", "score_playmaker_winger_adj", "score_inside_forward_adj"]].head()


,player_id,score_direct_runner_adj,score_playmaker_winger_adj,score_inside_forward_adj
0,2941,33.034062,22.283728,40.491437
1,2944,17.137427,13.947412,15.099593
2,2948,25.887095,16.501916,30.289737
3,2950,32.736022,42.003991,33.724630
4,2955,33.401783,44.136046,40.808751


In [37]:
base_cols = ["player_id"] + (["player_name"] if "player_name" in df.columns else []) + ["minutes", "age_cohort", "age_factor", "form_factor", "cluster_agreement"]
view_cols = base_cols + [
    "dribbles_attempted_per90", "dribbles_successful_per90", "dribble_success_rate",
    "carries_per90", "progressive_carries_per90", "passes_per90", "progressive_passes_per90",
    "shots_per90", "xg_per90",
    "score_direct_runner", "score_direct_runner_adj",
    "score_playmaker_winger", "score_playmaker_winger_adj",
    "score_inside_forward", "score_inside_forward_adj",
]

shortlist_direct = df.sort_values("score_direct_runner_adj", ascending=False)[view_cols].head(TOP_N).reset_index(drop=True)
shortlist_playmaker = df.sort_values("score_playmaker_winger_adj", ascending=False)[view_cols].head(TOP_N).reset_index(drop=True)
shortlist_inside = df.sort_values("score_inside_forward_adj", ascending=False)[view_cols].head(TOP_N).reset_index(drop=True)

shortlist_direct.head(10)


,player_id,player_name,minutes,age_cohort,age_factor,form_factor,cluster_agreement,dribbles_attempted_per90,dribbles_successful_per90,dribble_success_rate,...,passes_per90,progressive_passes_per90,shots_per90,xg_per90,score_direct_runner,score_direct_runner_adj,score_playmaker_winger,score_playmaker_winger_adj,score_inside_forward,score_inside_forward_adj
0,15470,Delphine Cascarino,1040.0,Unknown,1.0,1.089917,0.533333,6.836538,3.980769,0.582278,...,37.817308,3.548077,2.076923,0.082197,83.218011,48.373700,70.081711,40.737715,74.374131,43.232851
1,3385,Alexis Alejandro Sánchez Sánchez,6890.0,Unknown,1.0,1.089847,0.533333,3.670537,2.259797,0.615658,...,33.047896,6.087083,2.207547,0.238847,81.387344,47.306535,79.850487,46.413234,85.403338,49.640839
2,25729,Klara Bühl,910.0,Unknown,1.0,1.081043,0.533333,6.626374,3.263736,0.492537,...,33.923077,6.527473,2.571429,0.207345,81.860223,47.197029,78.496175,45.257467,83.758693,48.291604
3,8875,Ricardo Gabriel Álvarez,780.0,Unknown,1.0,1.059673,0.533333,3.692308,2.769231,0.750000,...,30.230769,9.230769,0.461538,0.004367,83.478790,47.178791,80.568498,45.534013,41.735049,23.586939
4,3499,Jean-Eric Maxim Choupo-Moting,3640.0,Unknown,1.0,1.086940,0.533333,4.697802,3.041209,0.647368,...,26.307692,3.931319,1.978022,0.167044,81.289986,47.123922,68.522253,39.722449,82.491307,47.820329
5,4926,Francisco Román Alarcón Suárez,1430.0,Unknown,1.0,1.074298,0.533333,3.335664,2.454545,0.735849,...,32.475524,4.909091,1.132867,0.100945,81.583797,46.744149,75.992698,43.540681,69.233310,39.667830
6,23660,Sambou Yatabaré,780.0,Unknown,1.0,1.041335,0.533333,4.153846,3.692308,0.888889,...,32.307692,12.000000,0.692308,0.130234,84.057719,46.683872,85.657163,47.572169,65.316412,36.275348
7,7156,Federico Chiesa,1170.0,Unknown,1.0,1.062191,0.533333,5.076923,2.846154,0.560606,...,30.076923,4.384615,3.692308,0.143065,82.268776,46.605395,72.001043,40.788708,82.682545,46.839796
8,8521,Daniel Caligiuri,3510.0,Unknown,1.0,1.084437,0.533333,4.897436,2.282051,0.465969,...,34.717949,6.743590,1.769231,0.116852,80.349444,46.471406,77.859875,45.031524,73.428373,42.468493
9,5571,Hirving Rodrigo Lozano Bahena,390.0,Unknown,1.0,1.005563,0.533333,3.692308,3.000000,0.812500,...,40.384615,12.461538,3.461538,0.142263,86.333449,46.300665,90.711057,48.648378,84.958275,45.563159


In [38]:
role_scores_out = DATA_DIR / "winger_role_scores_v2.parquet"
direct_out = DATA_DIR / "shortlist_direct_runner_v2.csv"
playmaker_out = DATA_DIR / "shortlist_playmaker_winger_v2.csv"
inside_out = DATA_DIR / "shortlist_inside_forward_v2.csv"

df.to_parquet(role_scores_out, index=False)
shortlist_direct.to_csv(direct_out, index=False)
shortlist_playmaker.to_csv(playmaker_out, index=False)
shortlist_inside.to_csv(inside_out, index=False)

print("Saved:")
print(" -", role_scores_out)
print(" -", direct_out)
print(" -", playmaker_out)
print(" -", inside_out)


Saved:
 - ..\data\processed\winger_role_scores_v2.parquet
 - ..\data\processed\shortlist_direct_runner_v2.csv
 - ..\data\processed\shortlist_playmaker_winger_v2.csv
 - ..\data\processed\shortlist_inside_forward_v2.csv


In [39]:
import pandas as pd

p = pd.read_parquet("../data/processed/winger_profiles.parquet")
r = pd.read_parquet("../data/processed/winger_role_scores_v2.parquet")

print("profiles shape:", p.shape)
print("profiles cluster_name counts:")
print(p["cluster_name"].value_counts(dropna=False))

print("\nrole_scores_v2 shape:", r.shape)
print("role_scores_v2 cluster_name counts:")
print(r["cluster_name"].value_counts(dropna=False))


profiles shape: (1438, 13)
profiles cluster_name counts:
cluster_name
1v1 Specialist             934
Inside Forward Finisher    504
Name: count, dtype: int64

role_scores_v2 shape: (1438, 59)
role_scores_v2 cluster_name counts:
cluster_name
1v1 Specialist             934
Inside Forward Finisher    504
Name: count, dtype: int64
